In [7]:
import torch.nn as nn
import torch

class ConvLSTMCell(nn.Module):

    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.padding = kernel_size[0] // 2, kernel_size[1] // 2
        self.bias = bias

        self.conv = nn.Conv2d(in_channels=self.input_dim + self.hidden_dim,
                              out_channels=4 * self.hidden_dim,
                              kernel_size=self.kernel_size,
                              padding=self.padding,
                              bias=self.bias)

    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state

        combined = torch.cat([input_tensor, h_cur], dim=1)  # concatenate along channel axis

        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)

        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)

        return h_next, c_next

    def init_hidden(self, batch_size, image_size):
        height, width = image_size
        return (torch.zeros(batch_size, self.hidden_dim, height, width, device=self.conv.weight.device),
                torch.zeros(batch_size, self.hidden_dim, height, width, device=self.conv.weight.device))


class ConvLSTM(nn.Module):

    def __init__(self, input_dim, hidden_dim, kernel_size, num_layers, channel_last=True,
                 batch_first=False, bias=True, return_all_layers=False):
        super(ConvLSTM, self).__init__()

        self._check_kernel_size_consistency(kernel_size)

        kernel_size = self._extend_for_multilayer(kernel_size, num_layers)
        hidden_dim = self._extend_for_multilayer(hidden_dim, num_layers)
        if not len(kernel_size) == len(hidden_dim) == num_layers:
            raise ValueError('Inconsistent list length.')

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.num_layers = num_layers
        self.batch_first = batch_first
        self.channel_last = channel_last
        self.bias = bias
        self.return_all_layers = return_all_layers

        cell_list = []
        bn_list = []
        relu_list = []
        for i in range(0, self.num_layers):
            cur_input_dim = self.input_dim if i == 0 else self.hidden_dim[i - 1]

            cell_list.append(ConvLSTMCell(input_dim=cur_input_dim,
                                          hidden_dim=self.hidden_dim[i],
                                          kernel_size=self.kernel_size[i],
                                          bias=self.bias))
            
            relu_list.append(nn.ReLU())
            bn_list.append(nn.BatchNorm3d(self.hidden_dim[i]))

        self.cell_list = nn.ModuleList(cell_list)
        self.relu_list = nn.ModuleList(relu_list)
        self.bn_list = nn.ModuleList(bn_list)

    def forward(self, input_tensor, hidden_state=None):
        if not self.batch_first:
            input_tensor = input_tensor.permute(1, 0, 2, 3, 4)

        if self.channel_last:
            input_tensor = input_tensor.permute(0, 1, 4, 2, 3)

        b, _, _, h, w = input_tensor.size()

        if hidden_state is not None:
            raise NotImplementedError()
        else:
            hidden_state = self._init_hidden(batch_size=b, image_size=(h, w))

        layer_output_list = []
        last_state_list = []
        seq_len = input_tensor.size(1)
        cur_layer_input = input_tensor

        for layer_idx in range(self.num_layers):
            h, c = hidden_state[layer_idx]
            output_inner = []
            for t in range(seq_len):
                h, c = self.cell_list[layer_idx](input_tensor=cur_layer_input[:, t, :, :, :], cur_state=[h, c])
                output_inner.append(h)

            layer_output = torch.stack(output_inner, dim=1)
            layer_output = self.relu_list[layer_idx](layer_output)
            layer_output = self.bn_list[layer_idx](layer_output)
            #layer_output = layer_output + cur_layer_input  # Apply residual connection
            cur_layer_input = layer_output

            layer_output_list.append(layer_output)
            last_state_list.append([h, c])

        if not self.return_all_layers:
            layer_output_list = layer_output_list[-1:]
            last_state_list = last_state_list[-1:]

        return layer_output_list, last_state_list

    def _init_hidden(self, batch_size, image_size):
        init_states = []
        for i in range(self.num_layers):
            init_states.append(self.cell_list[i].init_hidden(batch_size, image_size))
        return init_states

    @staticmethod
    def _check_kernel_size_consistency(kernel_size):
        if not (isinstance(kernel_size, tuple) or
                (isinstance(kernel_size, list) and all([isinstance(elem, tuple) for elem in kernel_size]))):
            raise ValueError('`kernel_size` must be tuple or list of tuples')

    @staticmethod
    def _extend_for_multilayer(param, num_layers):
        if not isinstance(param, list):
            param = [param] * num_layers
        return param

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchsummary import summary
from torchviz import make_dot
from torch.utils.data import DataLoader, TensorDataset, random_split
import torch.nn.functional as F
from torch.nn.functional import one_hot
from torch import Tensor
from typing import Union
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
from sklearn.metrics import roc_curve
import os

print("Should return True if a GPU is available: ",torch.cuda.is_available())  
print("Number of GPUs available: ",torch.cuda.device_count())  
print("Name of the first GPU: ",torch.cuda.get_device_name(0)) 

from sklearn.utils import class_weight
from sklearn.metrics import mean_squared_error, mean_absolute_error, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix
import scipy, scipy.ndimage
import pickle
import sys
import numpy as np
import pandas as pd
# Set the seed for reproducibility
# seed = int(sys.argv[6]) # 42, 0, 123, 999
seed = 42
print("SEED: ", seed)
torch.manual_seed(seed)

# Optionally set the seed for CUDA (if using GPU)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    
# Custom worker_init_fn to ensure each worker has a different seed
def worker_init_fn(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)
    
class PrecisionLoss2(nn.Module):
    def __init__(self, epsilon=1e-8, pos_weight=None):
        super(PrecisionLoss2, self).__init__()
        self.epsilon = epsilon
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        """
        logits: predicted scores from the model (before applying sigmoid)
        targets: ground truth binary labels (0 or 1)
        """
        # Apply sigmoid to get probabilities
        probs = torch.sigmoid(logits)
        
        if self.pos_weight is not None:
            # Apply class weight to the positive class
            targets = targets * self.pos_weight
        
        # Smooth approximation of true positives
        TP = torch.sum(probs * targets)
        
        # Smooth approximation of false positives
        FP = torch.sum(probs * (1 - targets))
        
        # Precision
        precision = TP / (TP + FP + self.epsilon)
        
        # Loss is defined as 1 - precision (to maximize precision, minimize the loss)
        loss = 1 - precision
        return loss

class FocalLoss2(nn.Module):
    """Computes the focal loss between input and target
    as described here https://arxiv.org/abs/1708.02002v2

    Args:
        gamma (float): The focal loss focusing parameter.
        weights (Union[None, Tensor]): Rescaling weight given to each class.
        If given, has to be a Tensor of size 2 for binary classification. optional.
        reduction (str): Specifies the reduction to apply to the output.
        It should be one of the following 'none', 'mean', or 'sum'.
        default 'mean'.
        ignore_index (int): Specifies a target value that is ignored and
        does not contribute to the input gradient. optional.
        eps (float): Smoothing to prevent log from returning inf.
    """
    def __init__(
            self,
            gamma: float,
            weights: Union[None, Tensor] = None,
            reduction: str = 'mean',
            ignore_index: int = -100,
            eps: float = 1e-16
            ) -> None:
        super().__init__()
        if reduction not in ['mean', 'none', 'sum']:
            raise NotImplementedError(
                'Reduction {} not implemented.'.format(reduction)
                )
        assert weights is None or isinstance(weights, Tensor), \
            'weights should be of type Tensor or None, but {} given'.format(
                type(weights))
        self.reduction = reduction
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.eps = eps
        self.weights = weights

    def forward(self, x: Tensor, target: Tensor) -> Tensor:
        assert torch.all((x >= 0.0) & (x <= 1.0)), ValueError(
            'The prediction values should be between 0 and 1, \
             make sure to pass the values to sigmoid for binary classification'
        )

        # Convert target to float
        target = target.float()

        # Compute the focal loss components
        bce_loss = nn.functional.binary_cross_entropy(x, target, reduction='none')
        pt = torch.where(target == 1, x, 1 - x)  # Probability of the true class
        focal_weight = (1 - pt) ** self.gamma

        # Apply class weights if provided
        if self.weights is not None:
            alpha_t = self.weights[1] * target + self.weights[0] * (1 - target)
            focal_weight = focal_weight * alpha_t

        loss = focal_weight * bce_loss

        # Handle ignore index
        if self.ignore_index >= 0:
            mask = target != self.ignore_index
            loss = loss * mask

        # Reduction
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

def get_neighbours(matrix,indices,distance):
    """
    matrix: numpy array
    indices: list with two integers
    distance: integer bigger than 1
    """
    # get indices of that element in the matrix
    indices = tuple(np.transpose(np.atleast_2d(indices)))
    # create a matrix of ones with same size as the input matrix
    dist = np.ones(np.shape(matrix))
    # put a zero at the location of the element we are looking at
    dist[indices] = 0
    # surround it by increasing numbers 1,2,3...
    dist = scipy.ndimage.distance_transform_cdt(dist, metric='chessboard')
    # get indeces of the values whoes distance is the requested (if we have more than 1, we have to add all previous)
    nb_indices = np.transpose(np.nonzero(dist == 1))
    for i in range(2,distance+1):
        nb_indices = np.concatenate((nb_indices,np.transpose(np.nonzero(dist == i))))
    return [matrix[tuple(ind)] for ind in nb_indices]


def get_custom_conf_matrix(orig,predi,dist=1):
    pred = predi.copy()
    for (i,j), value in np.ndenumerate(orig):
        if orig[i,j]==0 and pred[i,j]==0: # 0: true negative
            continue
        elif orig[i,j]==1 and pred[i,j]==1: # 1: true positive
            continue
        elif orig[i,j]==0 and pred[i,j]==1:
            if 1 in get_neighbours(orig,[i,j],dist):
                pred[i,j]=2 # 2: false positive with neighbouring positive
            else:
                pred[i,j]=3 # 3: false positive without neighbouring positive
        elif orig[i,j]==1 and pred[i,j]==0: # 4: false negative
            pred[i,j]=4

    result = pred.flatten().tolist()

    tn = result.count(0)
    tp = result.count(1)
    fp = result.count(3)
    fn = result.count(4)

    if tp==0 and result.count(2)==0:
        f1_orig = 0
        f1_new = 0
    elif tp==0:
        f1_orig = 0
        f1_new = (tp+result.count(2))/((tp+result.count(2))+0.5*(fp+fn))
    else:
        f1_orig = tp/(tp+0.5*((fp+result.count(2))+fn))
        f1_new = (tp+result.count(2))/((tp+result.count(2))+0.5*(fp+fn))

    if tp+fp+result.count(2) == 0:
        prec = 0
        prec_new = 0
    else:
        prec = tp/(tp+fp+result.count(2))
        prec_new = (tp+result.count(2))/(tp+result.count(2)+fp)
      
    if tp+fn==0:
        rec = 0
    else:  
        rec = tp/(tp+fn)
        
    if tp+result.count(2)+fn == 0:
        rec_new = 0
    else:
        rec_new = (tp+result.count(2))/(tp+result.count(2)+fn)

    return rec,prec,f1_orig,rec_new,prec_new,f1_new,pred

class ConvLSTMModel(nn.Module):
    def __init__(self, num_layers, input_chn, hidden_chn, kernel_size):
        super(ConvLSTMModel, self).__init__()
        self.num_layers = num_layers
    
        self.conv_lstm = ConvLSTM(input_chn, hidden_chn, kernel_size, num_layers, channel_last=True,
                 batch_first=True, bias=True, return_all_layers=True)
        self.final_conv = nn.Conv2d(in_channels=hidden_chn, out_channels=1, kernel_size=(3,3), padding='same')
        self.dropout = nn.Dropout(p=0.8)
    
    def forward(self, x):
        x = self.conv_lstm(x)[1][0][0] # selecting last_output, then extract sublist, then select h
        x = self.dropout(x)
        x = self.final_conv(x)
        return x

def train_model(model, train_loader, val_loader, criterions, optimizer, n_epoch, device, seed):
    criterion, criterion_precision, criterion_focal = criterions
    best_val_loss = +float('inf')
    model.train()
    
    # Define the scheduler
    scheduler = CosineAnnealingLR(optimizer, T_max=20)
    
    for epoch in range(n_epochs):
        running_loss = 0.0
        for data, target in train_loader:
            # move data to the GPU
            data, target = data.to(device), target.to(device)
            
            # zero the parameter gradients
            optimizer.zero_grad()
            
            # forward pass
            output = model(data)
            output = torch.squeeze(output, 1) # make size match
            loss = criterion(output, target)
            
            # backward pass and optimize
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        avg_train_loss = running_loss / len(train_loader)
        avg_val_loss, avg_rec, avg_prec, avg_f1, avg_rec_new, avg_prec_new, avg_f1_new = evaluate_model(model, val_loader, criterions, device, seed)
        
        print(f"Epoch {epoch}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
            
def evaluate_model(model, val_loader, criterions, device, seed):
    criterion, criterion_precision, criterion_focal = criterions
    model.eval()
    val_loss = 0.0
    all_targets = []
    all_predictions = []
    
    with torch.no_grad():
        for data, target in val_loader:
            
            data, target = data.to(device), target.to(device)
            
            output = model(data)
            output = torch.squeeze(output, 1) # make size match
            loss = criterion(output,target) 
            val_loss += loss.item()
            
            # Collecting predictions and targets for metric calculations
            all_targets.append(target.cpu().numpy())
            all_predictions.append(output.cpu().numpy())
    
    avg_val_loss = val_loss / len(val_loader)
    
    # Concatenate all predictions and targets
    all_targets = np.concatenate(all_targets)
    all_predictions = np.concatenate(all_predictions)
    
    # put model output logits into probabilities
    all_predictions = torch.sigmoid(torch.from_numpy(all_predictions)).numpy()

    # save prediction and truth before applying threshold
    os.makedirs("./output/maceio/predictions/", exist_ok=True)
    with open(f"./output/maceio/predictions/convlstm_predictions_maceio_{seed}.pkl", "wb") as f_p:  
        pickle.dump(all_predictions, f_p)
    with open(f"./output/maceio/predictions/convlstm_targets_maceio_{seed}.pkl", "wb") as f_t:  
        pickle.dump(all_targets, f_t)
    print("Saved final predictions and targets!")
    
    # apply thr to go from probability to class number
    thr=0.5
    all_predictions[all_predictions >= thr] = 1
    all_predictions[all_predictions < thr] = 0
    
    # Initialize lists to store metrics for each sample
    f1_list = []
    f1_new_list = []
    rec_list = []
    prec_list = []
    rec_new_list = []
    prec_new_list = []
    prec_auto = []
    rec_auto = []
    # Loop over each sample
    for i in range(np.shape(all_predictions)[0]):
        pred_flat = all_predictions[i,:] 
        true_flat = all_targets[i,:] 
        
        rec,prec,f1,rec_new,prec_new,f1_new,pred = get_custom_conf_matrix(true_flat,pred_flat,dist=1)
     
        rec_list.append(rec)
        prec_list.append(prec)
        f1_list.append(f1)
        rec_new_list.append(rec_new)
        prec_new_list.append(prec_new)
        f1_new_list.append(f1_new)
    
    # Compute the average precision, recall, and f1 score across all samples and classes
    avg_prec = np.mean(prec_list)
    avg_rec = np.mean(rec_list)
    avg_f1 = np.mean(f1_list)
    avg_prec_new = np.mean(prec_new)
    avg_rec_new = np.mean(rec_new)
    avg_f1_new = np.mean(f1_new)

    return avg_val_loss, avg_rec, avg_prec, avg_f1, avg_rec_new, avg_prec_new, avg_f1_new


def concat_all_cities(data_name,input_folder,ref_city,cities_list,crime_agg):
    x1 = np.load(f'./output/maceio/maceio_chrono.npz')[data_name]
    return x1

def select_features(mode,x_train,x_val):
    if mode == 'CS': # remove mobility channel(s) from data
        x_train = np.delete(x_train,[1,2,3,4,5,6,7,8,9,10,11,12],axis=4)
        x_val = np.delete(x_val,[1,2,3,4,5,6,7,8,9,10,11,12],axis=4)
        return (x_train,x_val)
    elif mode == 'CM':  # keep only C and M channels (so remove sociodem)
        x_train = x_train[:,:,:,:,:13]
        x_val = x_val[:,:,:,:,:13]
        return (x_train,x_val)
    elif mode == 'C':  # keep only C channel
        x_train = x_train[:,:,:,:,0]
        x_val = x_val[:,:,:,:,0]
        x_train = np.expand_dims(x_train, axis=-1)
        x_val = np.expand_dims(x_val, axis=-1)
        return (torch.from_numpy(x_train),torch.from_numpy(x_val))
    else:
        print("No valid mode selected, so we keep all the channels!")
        return (x_train,x_val)


def main(batch_size, n_epochs, input_folder, ref_city, cities_list, crime_agg, mode, num_layers, input_chn, hidden_chn, kernel_size, seed):
    # load the data as tensors
    x_train = torch.from_numpy(concat_all_cities(data_name='x_train',input_folder=input_folder,ref_city=ref_city,cities_list=cities_list,crime_agg=crime_agg)).float() 
    y_train = torch.from_numpy(concat_all_cities(data_name='y_train',input_folder=input_folder,ref_city=ref_city,cities_list=cities_list,crime_agg=crime_agg)).float()
    i_train = concat_all_cities(data_name='i_train',input_folder=input_folder,ref_city=ref_city,cities_list=cities_list,crime_agg=crime_agg)
    x_val = torch.from_numpy(concat_all_cities(data_name='x_val',input_folder=input_folder,ref_city=ref_city,cities_list=cities_list,crime_agg=crime_agg)).float()
    y_val = torch.from_numpy(concat_all_cities(data_name='y_val',input_folder=input_folder,ref_city=ref_city,cities_list=cities_list,crime_agg=crime_agg)).float()
    i_val = concat_all_cities(data_name='i_val',input_folder=input_folder,ref_city=ref_city,cities_list=cities_list,crime_agg=crime_agg)
    
    # Shuffle training data
    train_indices = np.random.permutation(x_train.shape[0])
    x_train = x_train[train_indices]
    y_train = y_train[train_indices]
    i_train = i_train[train_indices]

    # Shuffle validation data
    val_indices = np.random.permutation(x_val.shape[0])
    x_val = x_val[val_indices]
    y_val = y_val[val_indices]
    i_val = i_val[val_indices]
    
    # truncate to largest multiple of batch_size for consistent batching
    n_train = (x_train.shape[0] // batch_size) * batch_size
    n_val = (x_val.shape[0] // batch_size) * batch_size
    x_train = x_train[:n_train]
    y_train = y_train[:n_train]
    i_train = i_train[:n_train]
    x_val = x_val[:n_val]
    y_val = y_val[:n_val]
    i_val = i_val[:n_val]
    
    # Setting all NaN values to 0 (since they have been correctly identified already)
    x_train[np.isnan(x_train)] = 0
    y_train[np.isnan(y_train)] = 0
    x_val[np.isnan(x_val)] = 0
    y_val[np.isnan(y_val)] = 0
    
    # select features we want to use. Options: 'CS', 'CM', 'C', or nothing (so all features)
    x_train, x_val = select_features(mode=mode,x_train=x_train,x_val=x_val)
    
    print(f"Shapes: {np.shape(x_train)} {np.shape(y_train)}")
    print(f"Shapes: {np.shape(x_val)} {np.shape(y_val)}")
    
    # remove channel dimension so it will work at the end
    y_train = y_train.permute(0, 3, 1, 2)
    y_val = y_val.permute(0, 3, 1, 2)
    y_train = torch.squeeze(y_train, 1)
    y_val = torch.squeeze(y_val, 1)
    
    # define the device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # data loaders
    train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True,worker_init_fn=worker_init_fn)
    test_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=batch_size, shuffle=False)

    # Model
    model = ConvLSTMModel(num_layers, input_chn, hidden_chn, kernel_size)
    print(model)
    total_params = sum(p.numel() for p in model.parameters())
    print(f'Total number of parameters: {total_params}')  
    model.to(device) # to work on GPU
    
    # Calculate class weights
    targets_flat = y_train.view(-1)  # Flatten targets to (n_samples * 22 * 22)

    # Calculate the number of positive and negative examples
    num_positives = targets_flat.sum().item()
    num_negatives = targets_flat.size(0) - num_positives

    pos_weight = num_negatives / num_positives # Calculate pos_weight for the positive class

    # Convert pos_weight to a PyTorch tensor
    pos_weight_tensor = torch.tensor(pos_weight).float().to(device) # targets.device
    print("pos_weight_tensor: ",pos_weight_tensor)
    print([1, pos_weight_tensor.cpu().item()])
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor) 
    criterion_precision = PrecisionLoss2(pos_weight=pos_weight_tensor)
    criterion_focal = FocalLoss2(gamma=6,weights=torch.tensor([1, pos_weight_tensor.cpu().item()])) 
    criterions = (criterion, criterion_precision, criterion_focal)
    optimizer = optim.Adam(model.parameters(), lr=0.00001) 
    
    print("Training model...") 
    train_model(model, train_loader, test_loader, criterions, optimizer, n_epochs, device, seed)

    # make output folder if it does not exist
    os.makedirs("predictions", exist_ok=True)

    # save index of subgrids used
    with open(f"./output/maceio/predictions/convlstm_i_maceio__{seed}.pkl", "wb") as f:  
        pickle.dump(i_val, f)
    
if __name__ == "__main__":
    
    # params data loading
    input_folder = "./output/maceio"
    ref_city = "maceio"
    cities_list = None
    crime_agg = "all"
    mode = "C"
    seq_length = 8
    
    # params architecture model
    num_layers = 3
    hidden_chn = seq_length-1 #needs to match sequence legth (so LB-1)
    kernel_size = (3,3)

    if  mode == 'C':
        input_chn = 1
    elif mode == 'CM':
        input_chn = 13
    elif mode == 'CS':
        input_chn = 27
    else:
        input_chn = 39

    # params training
    batch_size = 55
    n_epochs = 200

    print(f"Data settings: ref_city={ref_city}, cities_list={cities_list}, crime_agg={crime_agg}, mode={mode}")
    print(f"Settings: n_epochs={n_epochs}, batch_size={batch_size}, num_layers={num_layers}, hidden_chn={hidden_chn}")
    main(batch_size, n_epochs, input_folder, ref_city, cities_list, crime_agg,mode, num_layers, input_chn, hidden_chn, kernel_size, seed)

Should return True if a GPU is available:  True
Number of GPUs available:  1
Name of the first GPU:  NVIDIA GeForce RTX 4070 Laptop GPU
SEED:  42
Data settings: ref_city=maceio, cities_list=None, crime_agg=all, mode=C
Settings: n_epochs=200, batch_size=55, num_layers=3, hidden_chn=7
Shapes: torch.Size([17985, 7, 16, 16, 1]) torch.Size([17985, 16, 16, 1])
Shapes: torch.Size([1925, 7, 16, 16, 1]) torch.Size([1925, 16, 16, 1])
ConvLSTMModel(
  (conv_lstm): ConvLSTM(
    (cell_list): ModuleList(
      (0): ConvLSTMCell(
        (conv): Conv2d(8, 28, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      )
      (1-2): 2 x ConvLSTMCell(
        (conv): Conv2d(14, 28, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      )
    )
    (relu_list): ModuleList(
      (0-2): 3 x ReLU()
    )
    (bn_list): ModuleList(
      (0-2): 3 x BatchNorm3d(7, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (final_conv): Conv2d(7, 1, kernel_size=(3, 3), stride=(1, 1), paddin

In [12]:
import numpy as np
import pickle
import sys
from sklearn.metrics import confusion_matrix
import scipy.ndimage
import os

def get_neighbours(matrix, indices, distance, mask=None):
    """
    matrix: numpy array
    indices: list with two integers
    distance: integer >= 1
    mask: optional mask (1 = valid, 0 = invalid)
    """
    indices = tuple(np.transpose(np.atleast_2d(indices)))
    dist = np.ones(np.shape(matrix))
    dist[indices] = 0

    dist = scipy.ndimage.distance_transform_cdt(dist, metric='chessboard')
    nb_indices = np.transpose(np.nonzero(dist == 1))
    for i in range(2, distance + 1):
        nb_indices = np.concatenate((nb_indices, np.transpose(np.nonzero(dist == i))))

    neighbours = []
    for ind in nb_indices:
        i, j = ind
        if 0 <= i < matrix.shape[0] and 0 <= j < matrix.shape[1]:
            if mask is None or mask[i, j] == 1:  # only use valid cells
                neighbours.append(matrix[i, j])
    return neighbours


def get_custom_conf_matrix(orig, predi, dist=1, mask=None):
    pred = predi.copy()
    for (i, j), value in np.ndenumerate(orig):
        if mask is not None and mask[i, j] == 0:
            continue  # skip masked cells 
        if orig[i, j] == 0 and pred[i, j] == 0:  # TN
            continue
        elif orig[i, j] == 1 and pred[i, j] == 1:  # TP
            continue
        elif orig[i, j] == 0 and pred[i, j] == 1:
            if 1 in get_neighbours(orig, [i, j], dist, mask=mask):
                pred[i, j] = 2  # FP with neighboring crime
            else:
                pred[i, j] = 3  # FP with no neighbor
        elif orig[i, j] == 1 and pred[i, j] == 0:  # FN
            pred[i, j] = 4

    result = pred[mask == 1].flatten().tolist() if mask is not None else pred.flatten().tolist()
    
    tn = result.count(0)
    tp = result.count(1)
    fp = result.count(3)
    fn = result.count(4)
    fp_nb = result.count(2)

    # F1 & Precision/Recall handling
    if tp == 0 and fp_nb == 0:
        f1_orig = f1_new = 0
    elif tp == 0:
        f1_orig = 0
        f1_new = (tp + fp_nb) / ((tp + fp_nb) + 0.5 * (fp + fn))
    else:
        f1_orig = tp / (tp + 0.5 * ((fp + fp_nb) + fn))
        f1_new = (tp + fp_nb) / ((tp + fp_nb) + 0.5 * (fp + fn))

    if tp + fp + fp_nb == 0:
        prec = prec_new = 0
    else:
        prec = tp / (tp + fp + fp_nb)
        prec_new = (tp + fp_nb) / (tp + fp_nb + fp)

    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    rec_new = (tp + fp_nb) / (tp + fp_nb + fn) if (tp + fp_nb + fn) > 0 else 0

    return rec, prec, f1_orig, rec_new, prec_new, f1_new, pred

def calculate_performance_per_threhold(seed):
    full_mask = np.load(f"./output/maceio/maceio_mask_final.npy") # i think mask is not necessary for these metrics

    with open(f"./output/maceio/predictions/convlstm_i_maceio__{seed}.pkl", 'rb') as f:
        subgrid_indices = pickle.load(f)

    with open(f"./output/maceio/predictions/convlstm_predictions_maceio_{seed}.pkl", 'rb') as file:
        data_pred = pickle.load(file)

    with open(f"./output/maceio/predictions/convlstm_targets_maceio_{seed}.pkl", 'rb') as file:
        data_orig = pickle.load(file)
        
    print("All data opened!")

    originals = data_orig
    predictions = data_pred

    assert len(subgrid_indices) == predictions.shape[0], "Mismatch between subgrid indices and number of samples"

    
    perf_list = []

    for thr in np.arange(0.50, 1.01, 0.01):
        rec_list, prec_list, f1_list = [], [], []
        rec_new_list, prec_new_list, f1_new_list = [], [], []

        for sample in range(predictions.shape[0]):
            origi = originals[sample]
            predi = predictions[sample]
            predi2 = (predi >= thr).astype(int)

            top_i, top_j = subgrid_indices[sample]
            mask_subgrid = full_mask[top_i:top_i + 16, top_j:top_j + 16] 
                
            rec, prec, f1, rec_new, prec_new, f1_new, _ = get_custom_conf_matrix(
                origi, predi2, dist=1, mask=mask_subgrid
            )

            rec_list.append(rec)
            prec_list.append(prec)
            f1_list.append(f1)
            rec_new_list.append(rec_new)
            prec_new_list.append(prec_new)
            f1_new_list.append(f1_new)

        perf_list.append([
            thr,
            np.mean(rec_list), np.mean(prec_list), np.mean(f1_list),
            np.mean(rec_new_list), np.mean(prec_new_list), np.mean(f1_new_list)
        ])

    os.makedirs("./output/maceio/perf_thrs", exist_ok=True)
    output_path = f"./output/maceio/perf_thrs/convlstm_perf_thrs_maceio_{seed}.pkl"
    with open(output_path, "wb") as fp:
        pickle.dump(perf_list, fp)
    print(f"Saved performance metrics to {output_path}")
    
calculate_performance_per_threhold(seed = 42)

All data opened!
Saved performance metrics to ./output/maceio/perf_thrs/convlstm_perf_thrs_maceio_42.pkl


In [13]:
import pickle
import pandas as pd
with open("./output/maceio/perf_thrs/convlstm_perf_thrs_maceio_42.pkl", "rb") as f:
    perf_data = pickle.load(f)
# Check what type it is and display it
print(type(perf_data))
if isinstance(perf_data, dict):
    for k, v in perf_data.items():
        print(f"{k}: {v}")
elif isinstance(perf_data, (pd.DataFrame, pd.Series)):
    display(perf_data)
else:
    display(perf_data)

<class 'list'>


[[np.float64(0.5),
  np.float64(0.7275127729413444),
  np.float64(0.05945873685573505),
  np.float64(0.10498733091120592),
  np.float64(0.863687371893187),
  np.float64(0.33133978958677957),
  np.float64(0.4540238389821231)],
 [np.float64(0.51),
  np.float64(0.7202800374228945),
  np.float64(0.060047882367642726),
  np.float64(0.10579365635264133),
  np.float64(0.8603239405258695),
  np.float64(0.33460729730176947),
  np.float64(0.4566211941202126)],
 [np.float64(0.52),
  np.float64(0.7133929007500436),
  np.float64(0.06091963715686352),
  np.float64(0.10706787803352105),
  np.float64(0.8566994591022703),
  np.float64(0.3381719381086579),
  np.float64(0.4592697294203741)],
 [np.float64(0.53),
  np.float64(0.7060119296288128),
  np.float64(0.0619440733281081),
  np.float64(0.10809053164398236),
  np.float64(0.8528935300112774),
  np.float64(0.3411269390607269),
  np.float64(0.46111740920547206)],
 [np.float64(0.54),
  np.float64(0.6981481527130877),
  np.float64(0.06250245972893728),
  

In [14]:
import pickle
import pandas as pd
with open("./output/maceio/perf_thrs/convlstm_perf_thrs_maceio_42.pkl", "rb") as f:
    perf_list = pickle.load(f)
df_perf = pd.DataFrame(perf_list, columns=[
    "threshold", "recall", "precision", "f1",
    "recall_spatial", "precision_spatial", "f1_spatial"
])
display(df_perf)
# Find best threshold by F1
best_row = df_perf.loc[df_perf["f1_spatial"].idxmax()]
print(f"\nBest threshold: {best_row['threshold']:.2f}")
print(f"  Recall (spatial):    {best_row['recall_spatial']:.4f}")
print(f"  Precision (spatial): {best_row['precision_spatial']:.4f}")
print(f"  F1 (spatial):        {best_row['f1_spatial']:.4f}")

,threshold,recall,precision,f1,recall_spatial,precision_spatial,f1_spatial
0,0.50,0.727513,0.059459,0.104987,0.863687,0.331340,0.454024
1,0.51,0.720280,0.060048,0.105794,0.860324,0.334607,0.456621
2,0.52,0.713393,0.060920,0.107068,0.856699,0.338172,0.459270
3,0.53,0.706012,0.061944,0.108091,0.852894,0.341127,0.461117
4,0.54,0.698148,0.062502,0.109019,0.849473,0.344300,0.463510
5,0.55,0.684338,0.062939,0.109282,0.844266,0.346385,0.464204
6,0.56,0.677821,0.063424,0.110410,0.840212,0.347883,0.464954
7,0.57,0.670948,0.064463,0.111874,0.836626,0.350470,0.466753
8,0.58,0.657182,0.064913,0.112281,0.830852,0.353701,0.468533
9,0.59,0.646118,0.066079,0.113521,0.825774,0.356682,0.470488



Best threshold: 0.65
  Recall (spatial):    0.7771
  Precision (spatial): 0.3755
  F1 (spatial):        0.4776
